In [5]:
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# 1. Cargar el dataset
ARCHIVO_CSV = "banco_transacciones.csv"
df = pd.read_csv(ARCHIVO_CSV)

# Limpiar posibles espacios al inicio/final de las columnas
df.columns = df.columns.str.strip()

# 2. Detección automática de columnas
posibles_categoria = [
    "Categoria_Transaccion",
    "categoria_nombre",
    "Categoria",
    "categoria_slug",
]
posibles_descripcion = [
    "Descripcion_Transaccion",
    "concepto",
    "comercio",
    "Descripcion",
]

col_categoria = next((c for c in posibles_categoria if c in df.columns), None)
col_descripcion = next(
    (c for c in posibles_descripcion if c in df.columns), None
)

if not col_categoria or not col_descripcion:
    raise KeyError(
        f"No se detectaron las columnas esperadas. Columnas en el CSV: {list(df.columns)}"
    )

print(
    f"Cargado exitosamente. Columna texto: '{col_descripcion}' | Columna categoría: '{col_categoria}'"
)

# 3. Diccionario de Mapeo
clasificacion = {
    # Alimentación
    "Centro comercial": "Alimentación",
    "Comida rápida": "Alimentación",
    "Comida Rapida": "Alimentación",
    "Supermercado": "Alimentación",
    "Restaurantes": "Alimentación",
    # Transporte
    "Taxi": "Transporte",
    "Bus": "Transporte",
    "Metro": "Transporte",
    "Metrobus": "Transporte",
    "Gasolina": "Transporte",
    "Mantenimiento": "Transporte",
    "Multas": "Transporte",
    # Salud
    "Seguros": "Salud",
    "Farmacia": "Salud",
    "Privada": "Salud",
    # Vivienda & Educación
    "Renta": "Vivienda",
    "Colegiaturas": "Educación",
    "Colegiatura": "Educación",
    # Ocio
    "Viajes": "Ocio",
    "Streaming": "Ocio",
    "Videojuegos": "Ocio",
    "Videojuegos Consola": "Ocio",
    "Salidas": "Ocio",
    "Ropa": "Ocio",
    "Zapatos de Tacón": "Ocio",
    "Zapatos de Tacon": "Ocio",
    "Bolsas": "Ocio",
    "Maquillaje": "Ocio",
    "Barbería": "Ocio",
    "Barberia": "Ocio",
    "Salón de Belleza": "Ocio",
    "Salon de Belleza": "Ocio",
    "Artículos Deportivos": "Ocio",
    "Articulos Deportivos": "Ocio",
    # Servicios
    "Agua": "Servicios",
    "Electricidad": "Servicios",
    "Gas": "Servicios",
    "Internet y Telefonía Hogar": "Servicios",
    "Internet y Telefonia Hogar": "Servicios",
    "Telefonía Móvil": "Servicios",
    "Telefonia Movil": "Servicios",
    "Internet": "Servicios",
    "Teléfono": "Servicios",
    # Deudas e Ingresos
    "Tarjetas de crédito": "Deudas",
    "Préstamos bancarios": "Deudas",
    "Buro de credito": "Deudas",
    "Impuestos": "Deudas",
    "Ahorro": "Ahorro",
    "Nómina/Ingresos": "Ingreso",
    "Nomina / Ingresos": "Ingreso",
    "Nomina/Ingresos": "Ingreso",
}

# 4. Mapeo y saneamiento de datos
df["Macro_categoria"] = df[col_categoria].map(clasificacion).fillna("Otros")
df[col_descripcion] = df[col_descripcion].fillna("Gasto Vario").astype(str)

# 5. Selección de X y y
X = df[col_descripcion]
y = df["Macro_categoria"]

# 6. División Train/Test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 7. Pipeline de Machine Learning (TF-IDF + Regresión Logística)
ml_pipeline = Pipeline(
    [
        (
            "convertidor_texto",
            TfidfVectorizer(ngram_range=(1, 2), strip_accents="unicode"),
        ),
        (
            "algoritmo_ia",
            LogisticRegression(random_state=42, max_iter=1000, C=1.0),
        ),
    ]
)

# 8. Entrenamiento
print("Entrenando el modelo clasificador...")
ml_pipeline.fit(X_train, y_train)

# 9. Evaluación
y_pred = ml_pipeline.predict(X_test)
print("\n--- Reporte de Evaluación del Modelo ---")
print(classification_report(y_test, y_pred))

# 10. Exportar el archivo binario para el Backend
joblib.dump(ml_pipeline, "modelo_clasificador_salud_financiera.pkl")
print(
    "\n¡Modelo exportado exitosamente como 'modelo_clasificador_salud_financiera.pkl'!"
)

Cargado exitosamente. Columna texto: 'concepto' | Columna categoría: 'categoria_nombre'
Entrenando el modelo clasificador...

--- Reporte de Evaluación del Modelo ---
              precision    recall  f1-score   support

Alimentación       1.00      1.00      1.00        48
   Educación       1.00      1.00      1.00        26
     Ingreso       1.00      1.00      1.00       153
        Ocio       1.00      1.00      1.00       306
       Otros       1.00      1.00      1.00        81
       Salud       1.00      1.00      1.00        51
   Servicios       1.00      1.00      1.00       125
  Transporte       1.00      1.00      1.00       185
    Vivienda       1.00      1.00      1.00        25

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000


¡Modelo exportado exitosamente como 'modelo_clasificador_salud_financiera.pkl'!
